<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_1_SLR/17_1_6_SLR_GeneralizationTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Linear Regression: The Generalization Test (Train/Test Split)

Author: Brad Sheese

---

## What This Notebook Is About

Every notebook in this module so far has answered one question about a fitted line: *how well does it describe the data we fit it on?* That's useful — but it isn't why we fit models.

The whole point of a regression model is to make predictions on **data we haven't seen yet**. A real-estate company doesn't fit a price model to *re-predict* houses that have already sold. They fit it to predict houses that are *about to* sell.

This notebook is about the question that separates "I fit a model" from "I fit a *useful* model":

> **Does my model work on data it has never seen?**

Answering that requires a small but non-negotiable discipline: **never evaluate a model on the same data you fit it on**. That discipline is called the **train/test split**, and it is the foundation of every evaluation technique you'll meet in this course.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import io
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

sns.set_style('whitegrid')
rng = np.random.default_rng(seed=42)

Every $R^2$ we've computed in this module has the same flaw: we asked the model *"how well do you predict the data I just showed you?"*

That is not a fair question. Imagine a professor who gives you the exam questions in advance with the answer key attached, then lets you write the exam with the key open on your desk. You'd get 100% — but your score says nothing about whether you *learned* anything.

That's what we've been doing:

1. Hand the model all the houses.
2. The model draws the line that minimizes squared errors *on those houses*.
3. We ask *"how low are the squared errors on those same houses?"* to compute $R^2$.
4. Shocker: the errors are as low as they could possibly be, because the model chose the line to make them low.

This is called **training error**, and it is always the most optimistic possible estimate of the model's true ability. A model can memorize every quirk of the training data (including noise) and look like a genius — while falling apart on new data.

For a simple straight line with only two parameters, the cheating problem is modest. But the discipline for fixing it is the same at any level of complexity, and we need to learn it now.

This is the same principle as *"Cook's Distance is a detective, not an executioner"* from 17_1_4: honest evaluation requires separating learning from assessment. The train/test split enforces this separation; Cook's Distance just diagnoses, it doesn't decide.

> **The fix:** Pretend some of our data doesn't exist. Fit the line using only the data we kept. Score it on the data we pretended didn't exist. *That* score is an honest estimate of how the model would do on data it has never seen.

---

## Section 2: The Train/Test Split

Scikit-learn gives us a single function: `train_test_split`. It takes our `X` and `y`, randomly sets aside some fraction of the rows as a **test set**, and returns four arrays.

In [ ]:
# Ames Housing — same loader as 17_0_5. We'll use it briefly to establish a baseline.
url = 'https://www.openintro.org/data/csv/ames.csv'
req = urllib.request.Request(url, headers={
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36',
    'Accept': 'text/csv,*/*'})
with urllib.request.urlopen(req) as resp:
    ames = pd.read_csv(io.BytesIO(resp.read()))

X = ames[['area']].to_numpy()                      # 2D, because sklearn demands it (see 17_1_1)
y = ames['price'].to_numpy() / 1000.0              # price in $ thousands

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
)

print(f'Total: {len(y)} homes')
print(f'Train: {len(y_train):>5} ({len(y_train)/len(y):.0%})')
print(f'Test:  {len(y_test):>5} ({len(y_test)/len(y):.0%})')

Let's unpack the arguments, because you will see this function call everywhere:

- **`X, y`** — the features and target.
- **`test_size=0.20`** — fraction of rows held out for testing. 20% is a common default.
- **`random_state=42`** — seed for the random shuffling. Fixing it makes the split *reproducible* so that you, your graders, and your coworkers all get the same split.

The returned arrays follow the convention `X_train, X_test, y_train, y_test`, **in that exact order**. Getting the order wrong is one of the most common sklearn bugs you will ever write.

> **The golden rule:** Once the split is done, `X_test` and `y_test` are locked away. Do not use them for fitting, for choosing between models, or for tuning any parameter. The *only* acceptable use of the test set is computing your final evaluation metrics after all modeling decisions are made. If you peek at test performance to make even one modeling decision, you have lost the test — it is no longer an honest holdout.

---

## Section 3: Fit on Train, Score on Both

Fit the line on the training set only. Then ask the model to score itself on *both* sets.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = model.score(X_train, y_train)
test_r2  = model.score(X_test,  y_test)

print(f'Train R^2:  {train_r2:.4f}')
print(f'Test  R^2:  {test_r2:.4f}')
print(f'Gap:        {train_r2 - test_r2:+.4f}')

The test $R^2$ is actually *slightly higher* than the training $R^2$. How is that possible? The model saw the training data, not the test data. Shouldn't it always do better on data it has seen?

**No — and this is important.** The train and test $R^2$ are random quantities that depend on which houses happened to end up in each set by the luck of the split. A small difference in either direction is normal. What matters is not the sign of the gap but its size: here the gap is negligible (under 0.04). The model performs about equally well on both sets.

Let's verify this by repeating the split many times and watching the gap bounce around.

In [ ]:
n_splits = 200
gaps = np.empty(n_splits)
test_scores = np.empty(n_splits)

for i in range(n_splits):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=i)
    m = LinearRegression().fit(X_tr, y_tr)
    test_scores[i] = m.score(X_te, y_te)
    gaps[i] = m.score(X_tr, y_tr) - m.score(X_te, y_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(test_scores, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(test_scores.mean(), color='darkorange', linewidth=2, linestyle='--',
                label=f'Mean = {test_scores.mean():.3f}')
axes[0].set_xlabel('Test $R^2$')
axes[0].set_ylabel('Number of splits')
axes[0].set_title(f'Test $R^2$ across {n_splits} random splits')
axes[0].legend()

axes[1].hist(gaps, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Train $R^2$ $-$ Test $R^2$')
axes[1].set_ylabel('Number of splits')
axes[1].set_title('Gap is centered near zero — no systematic overfit')

plt.tight_layout()
plt.show()

print(f'Mean gap: {gaps.mean():.4f}  (positive = train better than test)')
print(f'Std of gap: {gaps.std():.4f}')
print(f'Fraction of splits where test R^2 > train R^2: {(gaps < 0).mean():.1%}')

The gap between train and test $R^2$ is centered near zero, with a standard deviation of only about 0.02. Sometimes test beats train, sometimes the reverse — but the difference is tiny either way. **A simple straight line with two parameters cannot memorize much, so it cannot overfit.**

This is the baseline to keep in mind. If your model has very few parameters relative to the number of observations, train and test performance will be similar. The cheating problem only becomes visible when the model has enough flexibility to memorize the training data.

---

**YOUR TURN (8 min):** Modify the 200-split loop above to use `test_size=0.40` instead of `0.20`. Re-run it. How does the distribution of test $R^2$ change — is the mean higher or lower? Is the spread wider or narrower? Write one sentence explaining why a larger test fraction produces this result.

---

### Does fixing model specification improve generalization?

The revised 17_1_5 made a concrete claim: *a correctly-specified model generalizes better than a misspecified one.* From 17_1_3 we know `mpg ~ displacement` violates Linearity (U-shaped residuals). From 17_1_5, `mpg ~ 1/displacement` fixes it. Let's check whether fixing the shape actually helps on **new data**, or whether the violation only matters for inference.

In [ ]:
mpg = sns.load_dataset('mpg').dropna().reset_index(drop=True)

X_raw = mpg[['displacement']].to_numpy()
X_inv = (1.0 / mpg['displacement']).to_numpy().reshape(-1, 1)
y_mpg = mpg['mpg'].to_numpy()

print('Averaging train and test R² across 200 random 80/20 splits:')
print()

for label, X_m in [('Raw  (mpg ~ displacement)    ', X_raw),
                    ('Fixed (mpg ~ 1/displacement) ', X_inv)]:
    train_r2s, test_r2s = [], []
    for seed in range(200):
        Xtr, Xte, ytr, yte = train_test_split(X_m, y_mpg, test_size=0.20, random_state=seed)
        m = LinearRegression().fit(Xtr, ytr)
        train_r2s.append(m.score(Xtr, ytr))
        test_r2s.append(m.score(Xte, yte))
    gap = np.mean(train_r2s) - np.mean(test_r2s)
    print(f'{label}  train R²: {np.mean(train_r2s):.3f}   test R²: {np.mean(test_r2s):.3f}   gap: {gap:+.3f}')

print()
print('The correctly-specified model scores higher on new data because it captures')
print('the real relationship — not because it memorized more training points.')

---

## Section 4: Making Overfitting Visible

Neither two-parameter model above showed a meaningful train/test gap. Misspecification hurts the *level* of $R^2$ (wrong shape → lower ceiling) but not the gap — with only two parameters, neither model can memorize enough to overfit.

The gap explodes the moment we add **more features** — even just powers of one variable — because the model gains the flexibility to curve, wiggle, and eventually pass through every noisy training point.

**The setup.** We'll generate 30 synthetic data points from $y = \sin(x) + \text{noise}$. Then we'll fit polynomial models of increasing complexity (degree 1, 3, 10, 20) using only the training points. We'll score each model on the training data it saw, and on fresh test data it didn't.

In [ ]:
def true_f(x):
    return np.sin(x)

n_train, n_test = 30, 200

x_train = rng.uniform(0, 2 * np.pi, size=n_train)
y_train_syn = true_f(x_train) + rng.normal(0, 0.3, size=n_train)

x_test = rng.uniform(0, 2 * np.pi, size=n_test)
y_test_syn = true_f(x_test) + rng.normal(0, 0.3, size=n_test)

Xtr = x_train.reshape(-1, 1)
Xte = x_test.reshape(-1, 1)

print(f'Training points: {n_train}')
print(f'Test points:      {n_test}  (more test points = more stable estimate of test error)')

### What are polynomial features?

A degree-$d$ polynomial model uses not just $x$ but $x, x^2, x^3, \ldots, x^d$ as predictors. The model equation becomes:

$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$$

When $d = 1$, this is just the straight line we already know. When $d = 10$, it has 11 parameters — enough to start bending wildly. Here's what the data actually looks like after expanding the features:

In [ ]:
poly_deg3 = PolynomialFeatures(degree=3)
X_demo = poly_deg3.fit_transform(Xtr[:5])  # first 5 training points

print("Original x values (first 5):")
print(np.round(x_train[:5], 3))
print()
print("Design matrix for degree-3 polynomial (first 5 rows):")
print("Columns: intercept, x, x^2, x^3")
print(np.round(X_demo, 4))

Each row in that matrix is what the model actually sees: the original $x$ plus its powers. More columns = more parameters = more flexibility to fit (and overfit) the training data. Now let's fit and visualize.

In [ ]:
degrees_to_plot = [1, 3, 10, 20]
x_grid = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, d in zip(axes.flatten(), degrees_to_plot):
    poly_model = make_pipeline(PolynomialFeatures(degree=d), LinearRegression())
    poly_model.fit(Xtr, y_train_syn)

    tr_score = poly_model.score(Xtr, y_train_syn)
    te_score = poly_model.score(Xte, y_test_syn)

    ax.plot(x_grid, true_f(x_grid), color='black', linestyle='--', linewidth=1.5,
            label='True function sin(x)')
    ax.scatter(x_train, y_train_syn, s=45, color='steelblue', alpha=0.7,
               edgecolor='white', zorder=3, label=f'Train ({n_train} points)')
    ax.plot(x_grid, poly_model.predict(x_grid), color='darkorange', linewidth=2.5,
            label=f'Degree-{d} fit')

    ax.set_ylim(-3, 3)
    ax.set_title(f'Degree {d}:  train $R^2$ = {tr_score:.2f},   test $R^2$ = {te_score:.2f}')
    ax.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

Read these four panels left-to-right, top-to-bottom:

- **Degree 1 (a straight line).** Can't bend at all. Both train and test $R^2$ are low. This is **underfitting**: the model is too simple to capture the real structure.
- **Degree 3.** A nice smooth cubic that hugs the true sine curve. Train and test scores are both decent. This is the sweet spot.
- **Degree 10.** The orange curve starts to wriggle. The model is bending toward individual noisy training points. Train score has climbed, but test score has started to fall.
- **Degree 20.** The orange line goes **wild**. It passes through almost every training point perfectly (train $R^2$ near 1.0) but makes garbage predictions on unseen data. This is **overfitting**: the model has memorized the training data, noise and all.

Now let's see the full picture by sweeping all degrees from 1 to 20.

In [ ]:
degrees = np.arange(1, 21)
train_scores, test_scores = [], []

for d in degrees:
    m = make_pipeline(PolynomialFeatures(degree=d), LinearRegression())
    m.fit(Xtr, y_train_syn)
    train_scores.append(m.score(Xtr, y_train_syn))
    test_scores.append(m.score(Xte, y_test_syn))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(degrees, train_scores, 'o-', color='steelblue', linewidth=2,
        label='Train $R^2$')
ax.plot(degrees, test_scores, 'o-', color='darkorange', linewidth=2,
        label='Test $R^2$')
ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.6,
           label='$R^2 = 0$ (predicting the mean)')
ax.set_xlabel('Polynomial degree (model complexity)')
ax.set_ylabel('$R^2$')
ax.set_xticks(degrees)
ax.set_ylim(-1.5, 1.05)
ax.set_title('Training $R^2$ climbs to 1.0; test $R^2$ peaks then collapses')
ax.legend()

# Annotate where test scores fall off the bottom of the clipped axis.
min_test = min(test_scores)
if min_test < -1.5:
    worst_deg = degrees[int(np.argmin(test_scores))]
    ax.annotate(f'continues to fall\n(min ≈ {min_test:.0f} at degree {worst_deg})',
                xy=(worst_deg, -1.45), xytext=(worst_deg - 5, -1.1),
                arrowprops=dict(arrowstyle='->', color='darkorange'),
                color='darkorange', fontsize=9)

plt.show()

best_idx = int(np.argmax(test_scores))
worst_idx = int(np.argmin(test_scores))
print(f'Best test R^2 at degree {degrees[best_idx]}  (R^2 = {test_scores[best_idx]:.3f})')
print(f'Worst test R^2 at degree {degrees[worst_idx]} (R^2 = {test_scores[worst_idx]:.3f})')

**This plot captures a central idea in machine learning.**

- The **blue curve** (train $R^2$) only ever goes up. More complexity *always* improves training performance.
- The **orange curve** (test $R^2$) rises, peaks around degree 3–5, and then **falls off a cliff**. At degree 20, it goes *negative*.

### Wait — $R^2$ can be negative?

Yes. On training data, $R^2$ is always between 0 and 1 because OLS *always* beats or ties the mean-predictor on the data it was fit on. But on **unseen test data**, the fitted line is no longer optimal for that set. A severely overfit model can do *worse than just guessing the mean*. When that happens, $R^2 < 0$.

> **A negative test $R^2$ is the clearest possible signal that you have overfit.** Your complex model is less useful than a horizontal line at $\bar{y}$.

---

**YOUR TURN (10 min):** Re-run the synthetic experiment with `n_train = 100` instead of 30. How does having more training data change the overfitting picture? Does the peak test $R^2$ shift? Does the degree-20 fit still go negative?

---

## Section 5: The Bias–Variance Tradeoff

Everything in Section 4 is an instance of the **bias–variance tradeoff**, one of the most important ideas in machine learning. It decomposes a model's expected error on new data into three pieces:

$$\text{Expected Test Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

- **Bias:** error from oversimplifying. A straight line through $\sin(x)$ is systematically wrong — it has high bias.
- **Variance:** error from being too sensitive to the training data. A degree-20 polynomial changes wildly if you change which 30 points you sample — it has high variance.
- **Irreducible error:** noise in the data itself. No model can remove it.

Let's measure bias and variance empirically by fitting 200 models at each degree, each on a *different* random training set. The resulting plot uses a **symlog** (symmetric log) y-axis: like a regular log scale, but with a small linear region near zero so values very close to 0 can be shown without going to $-\infty$. This lets bias and variance — which span many orders of magnitude — fit on the same axis.

In [ ]:
# Across many repeated samples, estimate bias^2 and variance at each degree.
# We cap at degree 12 because higher degrees with only 30 training points produce
# numerically extreme predictions that visually swamp the informative lower range.
n_reps = 200
x_test_fine = np.linspace(0, 2 * np.pi, 400)
Xte_fine = x_test_fine.reshape(-1, 1)

bias_sq_by_degree = []
variance_by_degree = []
bv_degrees = np.arange(1, 13)

for d in bv_degrees:
    preds = np.empty((n_reps, len(x_test_fine)))
    for i in range(n_reps):
        xs = rng.uniform(0, 2 * np.pi, size=n_train)
        ys = true_f(xs) + rng.normal(0, 0.3, size=n_train)
        m = make_pipeline(PolynomialFeatures(degree=d), LinearRegression())
        m.fit(xs.reshape(-1, 1), ys)
        preds[i] = m.predict(Xte_fine).flatten()

    avg_pred = preds.mean(axis=0)
    bias_sq = ((avg_pred - true_f(x_test_fine)) ** 2).mean()
    variance = preds.var(axis=0).mean()
    bias_sq_by_degree.append(bias_sq)
    variance_by_degree.append(variance)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(bv_degrees, bias_sq_by_degree, 'o-', color='red', linewidth=2, label='Bias$^2$')
ax.plot(bv_degrees, variance_by_degree, 'o-', color='green', linewidth=2, label='Variance')
ax.plot(bv_degrees, np.array(bias_sq_by_degree) + np.array(variance_by_degree),
        'o-', color='black', linewidth=2, label='Bias$^2$ + Variance = expected test error')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('Error (symlog scale)')
ax.set_xticks(bv_degrees)
ax.set_yscale('symlog', linthresh=0.01)
ax.set_title('Bias–variance decomposition: bias falls, variance rises (degrees 1–12)')
ax.legend()
plt.show()

The story in one plot:

- **Bias$^2$ (red)** starts high (a line through $\sin(x)$ is very wrong) and drops fast, leveling off near zero by degree 3–5.
- **Variance (green)** starts near zero (a line barely changes when you resample) and then explodes as degree increases.
- **Their sum (black)** — the expected test error — is U-shaped. The minimum is where bias and variance balance, which is exactly the degree that gave the best test $R^2$ in the sweep above.

This U-shape is not specific to polynomials. It holds for every model you will ever use: decision trees, neural networks, ridge regression, random forests. The art of model-building is finding the complexity level where bias and variance cross.

The repeated-sampling idea here is the same one from 17_1_2 — the bootstrap resampled pairs to measure slope uncertainty; here we resample entire training sets to measure bias and variance.

### The Three Regimes — a Reference Card

| Regime | Symptom | Fix |
|---|---|---|
| **Underfitting** (high bias) | Train and test both bad | More complex model: more features, higher degree |
| **Goldilocks zone** | Train decent, test close to train | You found it. Stop here. |
| **Overfitting** (high variance) | Train near 1.0, test much worse | Simplify, get more data, or use regularization |

The **train/test split** is the diagnostic tool that tells you which regime you're in. Without it, you literally cannot tell overfitting from success — on the training data they look identical.

> **The gap between train and test performance is your overfitting detector.** Small gap and both high? Fine. Small gap and both low? Underfit — increase complexity. Large gap with train high and test low? Overfit — simplify or get more data.

---

## Putting It All Together (End of the SLR Arc)

This is the last notebook in the Simple Linear Regression module. Here is the full journey:

| Notebook | Question answered |
|---|---|
| 17_0_1 – 17_0_5 | What *is* a regression line, mathematically? |
| 17_1_1 | How do sklearn and statsmodels fit it? |
| 17_1_2 | Is my slope real, or did I get lucky? |
| 17_1_3 | Are the LINE assumptions holding? |
| 17_1_4 | Is a single bad row wrecking my fit? |
| 17_1_5 | What if the shape is wrong? (transformations) |
| **17_1_6** | **Does my model work on new data?** (train/test split, overfitting) |

And the three-line pattern you can now apply to any model:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
honest_r2 = model.score(X_test, y_test)
```

The regime table from Section 5 describes every model you will ever build — keep it close:

| Regime | Symptom | Fix |
|---|---|---|
| **Underfitting** | Train and test both bad | More complex model |
| **Goldilocks zone** | Train decent, test close to train | You found it. Stop here. |
| **Overfitting** | Train near 1.0, test much worse | Simplify, get more data, or regularize |

### Where We're Going Next

A single train/test split is a good start, but it has limitations: the gap estimate depends on *which* 20% you held out. More reliable estimates come from **cross-validation**, which uses multiple train/test splits on the same dataset. And when models have many features, **regularization** (ridge, lasso) can directly penalize complexity to control the bias–variance tradeoff.

That is exactly what comes next in **17_2 (Multiple Linear Regression)**: cross-validation, regularization, and many predictors at once. Every tool you learned here — OLS, assumptions, influence, transformations, train/test splits — generalizes directly to many predictors. The math stays the same. The risks get bigger. The tools get sharper.

---

**FULL WORKFLOW CHALLENGE (20 min):** Load the Palmer Penguins dataset: `penguins = sns.load_dataset('penguins').dropna()`. Fit `body_mass_g ~ flipper_length_mm` and apply everything you've learned:

1. Check the statistical significance of the slope (p-value, CI).
2. Check the LINE assumptions (residual plot, Q-Q plot).
3. Check for influential points (Cook's Distance).
4. If assumptions are violated, try a transformation (log).
5. Split the data into train/test and compare train vs. test $R^2$.
6. Write 2–3 sentences: Is flipper length a good predictor of body mass? Which assumptions held? Were there influential points?

---
See you in 17_2.